In [10]:
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# KHỞI TẠO ĐƯỜNG DẪN & THƯ MỤC
# ------------------------------------------------------------------------------
DATA_DIR, PROCESSED_DIR = ".", "./processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Khai báo chính xác các file Inputs riêng biệt
F_RESEARCH  = "Research_Project_Data.xlsx"
F_CANDIDATE = "Candidate_Locations.xlsx"
F_HAVERSINE = "Distance_Matrix_Haversine.xlsx"

# ============================================================
# BƯỚC 1: Load sheets
# ============================================================
print("=" * 60 + "\nBƯỚC 1: Load sheets\n" + "=" * 60)

idt = {"DestinationRef": str, "ShipToID": str, "GroupID": str}

# 1. Load các sheet từ Research_Project_Data.xlsx
outbound   = pd.read_excel(f"{DATA_DIR}/{F_RESEARCH}", sheet_name="Outbound", dtype=idt)
branch_mst = pd.read_excel(f"{DATA_DIR}/{F_RESEARCH}", sheet_name="BranchMaster", dtype=idt)
cost_curve = pd.read_excel(f"{DATA_DIR}/{F_RESEARCH}", sheet_name="CostCurve")
current    = pd.read_excel(f"{DATA_DIR}/{F_RESEARCH}", sheet_name="CurrentScenario")

# 2. Load file Candidate Locations riêng biệt
if F_CANDIDATE.endswith(".csv"):
    candidates = pd.read_csv(f"{DATA_DIR}/{F_CANDIDATE}", dtype=str)
else:
    candidates = pd.read_excel(f"{DATA_DIR}/{F_CANDIDATE}", dtype=str)

# In log thông số các file/sheet đúng chuẩn format mẫu
print(f"  Outbound         {outbound.shape}")
print(f"  BranchMaster     {branch_mst.shape}")
print(f"  CostCurve        {cost_curve.shape}")
print(f"  CurrentScenario  {current.shape}")
print(f"  Candidates       {candidates.shape}")

# Gom nhóm Outbound theo Chi nhánh
agg = outbound.groupby("DestinationRef").agg(total_cbm=("BranchDeliveryVolumeInCBM", "sum")).reset_index()
branches = agg.merge(branch_mst[["DestinationRef", "Latitude", "Longitude", "State"]], on="DestinationRef", how="left")

# Lấy thông số kho gốc NSW-DC01
cur_row = current[current["Facility"].astype(str).str.contains("DC01", na=False)].iloc[0]
I_BAR  = float(cur_row["AverageInventoryInCBM"])
S_RATE = float(cur_row["StorageCostPerCBM (AUD)"])
H_RATE = float(cur_row["HandlingCostPerCBM (AUD)"])

# Đóng gói danh sách Candidate DCs (87 ứng viên)
dc_meta = candidates[["DestinationRef", "State", "Latitude", "Longitude"]].drop_duplicates("DestinationRef").set_index("DestinationRef")
dc_meta.index.name = "dc_id"

# Thêm NSW-DC01 vào danh sách nếu chưa có (87 -> 88 candidates)
if "NSW-DC01" not in dc_meta.index:
    dc_meta.loc["NSW-DC01"] = {
        "State": "New South Wales",
        "Latitude": float(cur_row["Latitude"]),
        "Longitude": float(cur_row["Longitude"])
    }

print(f"  Branches: {len(branches)}")
print(f"  Candidates: {len(dc_meta)}")
print(f"  I_BAR={I_BAR:,.0f}, S_RATE={S_RATE:.1f}, H_RATE={H_RATE:.1f}")
print("✓ BƯỚC 1 xong!")

# ============================================================
# BƯỚC 2: Haversine & Cost Matrix
# ============================================================
print("\n" + "=" * 60 + "\nBƯỚC 2: Haversine\n" + "=" * 60)

# Tải ma trận Haversine có sẵn từ file Distance_Matrix_Haversine HOẶC tự tính bằng tọa độ
if os.path.exists(f"{DATA_DIR}/{F_HAVERSINE}"):
    if F_HAVERSINE.endswith(".csv"):
        dist_df = pd.read_csv(f"{DATA_DIR}/{F_HAVERSINE}", index_col=0)
    else:
        dist_df = pd.read_excel(f"{DATA_DIR}/{F_HAVERSINE}", index_col=0)
    # Chuẩn hóa tên cột của dist_df để loại bỏ tiền tố 'Cand_' nếu có
    dist_df.columns = dist_df.columns.str.replace("^Cand_", "", regex=True)

    # Nếu thiếu cột NSW-DC01, tiến hành tính toán và thêm vào ma trận khoảng cách
    if "NSW-DC01" not in dist_df.columns:
        lat1, lon1 = np.radians(branches["Latitude"].values.astype(float)), np.radians(branches["Longitude"].values.astype(float))
        lat2, lon2 = np.radians(float(dc_meta.loc["NSW-DC01", "Latitude"])), np.radians(float(dc_meta.loc["NSW-DC01", "Longitude"]))
        a = np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2
        nsw_dist = 2 * 6371.0088 * np.arcsin(np.sqrt(a))
        dist_df["NSW-DC01"] = nsw_dist
else:
    # Tính Haversine trực tiếp nếu chưa có file ma trận xuất sẵn
    lat1, lon1 = np.radians(branches["Latitude"].values[:, None].astype(float)), np.radians(branches["Longitude"].values[:, None].astype(float))
    lat2, lon2 = np.radians(dc_meta["Latitude"].values[None, :].astype(float)), np.radians(dc_meta["Longitude"].values[None, :].astype(float))
    a = np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2
    dist_matrix = 2 * 6371.0088 * np.arcsin(np.sqrt(a))
    dist_df = pd.DataFrame(dist_matrix, index=branches["DestinationRef"], columns=dc_meta.index)

print(f"  dist shape = {dist_df.shape}")
print(f"  Max dist = {dist_df.values.max():,.1f} km")

# Áp dụng Cost Curve và State Fallback
FALLBACK = {
    "Australian Capital Territory": "New South Wales",
    "South Australia": "Victoria",
    "Tasmania": "Victoria",
    "Northern Territory": "Queensland"
}
dc_state = dc_meta["State"].to_dict()

curve = {}
for r in cost_curve.itertuples(index=False):
    curve.setdefault(r.State, []).append((r.Mode, float(r[2]), float(r[3]), float(r[4]), float(r[5])))
for s in curve:
    curve[s].sort(key=lambda x: x[3])

C = np.zeros(dist_df.shape)
MODE = np.empty(dist_df.shape, dtype=object)

for j, dc in enumerate(dist_df.columns):
    st = FALLBACK.get(dc_state[dc], dc_state[dc])
    bands = curve[st]
    d = dist_df.values[:, j]
    ic, sl = np.full_like(d, bands[-1][1]), np.full_like(d, bands[-1][2])
    for k, (mode, intercept, slope, lo, hi) in enumerate(bands):
        m = (d >= lo) if (k == len(bands) - 1) else ((d >= lo) & (d < hi))
        ic[m], sl[m], MODE[m, j] = intercept, slope, mode
    C[:, j] = ic + sl * d

cost_df = pd.DataFrame(C, index=branches["DestinationRef"], columns=dist_df.columns)
ftl_share = (MODE == "FTL").mean()

print(f"  cost shape = {cost_df.shape}, FTL share = {ftl_share:.1%}")

# Kiểm tra sai số Baseline chuẩn
EXPECTED = 947297.15
base_matrix = float(cost_df["NSW-DC01"].sum())
err = abs(base_matrix - EXPECTED) / EXPECTED * 100

print(f"\n  Baseline: {base_matrix:,.2f} | expected {EXPECTED:,.2f} | err {err:.6f}%")

# Lưu 4 file chuẩn hóa vào thư mục processed/
dist_df.to_csv(f"{PROCESSED_DIR}/distance_matrix_km.csv")
cost_df.to_csv(f"{PROCESSED_DIR}/cost_matrix_base.csv")
branches.to_csv(f"{PROCESSED_DIR}/branch_demand.csv", index=False)
dc_meta.to_csv(f"{PROCESSED_DIR}/candidates.csv")

print(f"\n  ✓ Saved 4 files to processed/")
print("\n✓ BƯỚC 2 xong!")

BƯỚC 1: Load sheets
  Outbound         (145490, 9)
  BranchMaster     (635, 10)
  CostCurve        (8, 6)
  CurrentScenario  (3, 7)
  Candidates       (87, 18)
  Branches: 635
  Candidates: 88
  I_BAR=14,898, S_RATE=30.0, H_RATE=10.0
✓ BƯỚC 1 xong!

BƯỚC 2: Haversine
  dist shape = (635, 88)
  Max dist = 3,793.6 km
  cost shape = (635, 88), FTL share = 7.3%

  Baseline: 947,297.15 | expected 947,297.15 | err 0.000000%

  ✓ Saved 4 files to processed/

✓ BƯỚC 2 xong!


In [11]:
import types
D = types.SimpleNamespace()

In [12]:
import numpy as np
import pandas as pd

print("STEP 3: SHARED EVALUATOR")


class SharedEvaluator:
    """
    Evaluates a set S of open DCs.
    Branches are always assigned to their cheapest open DC (same rule for
    every algorithm), so results are comparable across methods.

    Weights never change the assignment (argmin_j w_i*c_ij = argmin_j c_ij);
    they only change the delivery objective, which also decides which S the
    MILP / heuristic picks.

    Use evaluate() for final solutions (full validation + all metrics).
    Use fast_delivery() inside heuristic loops (numpy only, no pandas).
    """

    def __init__(self, cost_matrix, dist_matrix, demand_series,
                 i_bar, s_rate, h_rate,
                 w_svc=None, cbm_for_handling=None):
        self.C = cost_matrix
        self.D = dist_matrix

        # One-time data checks
        if not self.C.index.equals(self.D.index) or not self.C.columns.equals(self.D.columns):
            raise ValueError("cost and distance matrices must share the same index and columns")
        if not np.isfinite(self.D.values).all():
            raise ValueError("Bad distances (NaN/Inf)")
        if (self.D.values < 0).any():
            raise ValueError("Negative distances")
        if not np.isfinite(self.C.values).all():
            raise ValueError("Bad costs (NaN/Inf)")
        if (self.C.values < 0).any():
            raise ValueError("Negative costs")
        if not self.C.index.is_unique:
            raise ValueError("Duplicate branch ids in cost matrix")
        if not self.C.columns.is_unique:
            raise ValueError("Duplicate DC ids in cost matrix")

        # Align every per-branch series to the cost matrix index
        def _align(series, name):
            if series is None:
                return None
            missing = set(self.C.index) - set(series.index)
            if missing:
                raise ValueError(f"{name} missing for {len(missing)} branches, e.g. {list(missing)[:3]}")
            s = series.reindex(self.C.index)
            if s.isna().any():
                raise ValueError(f"NaN in {name}")
            return s.astype(float)

        # demand is required
        self.q = _align(demand_series, "demand_series")
        if self.q is None:
            raise ValueError("demand_series is required")

        # Service weights and handling CBM default to demand, but can differ
        self.w_svc = _align(w_svc, "w_svc") if w_svc is not None else self.q
        self.cbm_handling = (_align(cbm_for_handling, "cbm_for_handling")
                             if cbm_for_handling is not None else self.q)

        self.I_bar = float(i_bar)
        self.s = float(s_rate)
        self.h = float(h_rate)
        self.Q = float(self.w_svc.sum())

        # validate service weights
        if (self.w_svc < 0).any() or not self.Q > 0:
            raise ValueError("w_svc must be non-negative with positive sum")

        self.handling_cost = self.h * float(self.cbm_handling.sum())  # independent of S

        self._C_vals = self.C.values
        self._dist_vals = self.D.values
        self._dist_cols = self.D.columns
        self._col_pos = {c: i for i, c in enumerate(self.C.columns)}

    # ------------------------------------------------------------------
    def storage_cost(self, K):
        """Storage = s * I_bar * sqrt(K). More DCs -> LESS pooling -> HIGHER cost."""
        return self.s * self.I_bar * np.sqrt(K)

    def make_weights(self, w_obj=None):
        """Objective weights as a numpy array with mean 1 (Section 9.2). None -> all ones."""
        if w_obj is None:
            return np.ones(len(self.C.index))
        w = w_obj.reindex(self.C.index)
        if w.isna().any():
            raise ValueError("w_obj has missing branches")
        if (w < 0).any() or w.mean() <= 0:
            raise ValueError("w_obj must be non-negative with positive mean")
        return (w / w.mean()).values

    def _check_open_set(self, S, K):
        if len(S) != K or len(set(S)) != K:
            raise ValueError(f"Expected {K} distinct DCs, got {len(set(S))}: {S}")
        unknown = set(S) - set(self._col_pos)
        if unknown:
            raise ValueError(f"Open DC not in candidate set: {unknown}")

    def _validate_solution(self, S, K, assignments):
        """Feasibility checks (Section 15.2)."""
        self._check_open_set(S, K)
        if len(assignments) != len(self.C):
            raise ValueError(f"Not every branch assigned: {len(assignments)} vs {len(self.C)}")
        if not assignments.index.is_unique:
            raise ValueError("Duplicate branch assignment")
        if assignments.isna().any():
            raise ValueError("NaN in assignment")
        if not set(assignments.unique()).issubset(set(S)):
            raise ValueError(f"Branch assigned to closed DC: {set(assignments.unique()) - set(S)}")

    # ------------------------------------------------------------------
    def fast_delivery(self, S, wn=None):
        """
        Delivery objective Z(S) = sum_i wn_i * min_{j in S} c_ij, numpy only.
        For heuristic loops. wn = make_weights(w_obj) (None -> equal-frequency).
        """
        S = list(S)
        self._check_open_set(S, len(S))
        lane = self._C_vals[:, [self._col_pos[j] for j in S]].min(axis=1)
        return float(lane.sum() if wn is None else (lane * wn).sum())

    # ------------------------------------------------------------------
    def evaluate(self, S, K=None, w_obj=None, quiet=True):
        """
        S     : list of open DC ids
        K     : number of open DCs (defaults to len(S); if given, must match)
        w_obj : optional Series of branch weights for the delivery objective
                (None = equal-frequency proxy, S0)
        Returns (metrics dict, assignments Series branch -> dc_id)

        delivery_cost           : equal-frequency lane-cost proxy (always)
        delivery_cost_weighted  : normalized weighted proxy (= delivery_cost if w_obj is None)
        delivery_objective      : the scenario's objective (weighted if w_obj given)
        total_modeled_cost      : delivery_objective + storage + handling (follows the scenario)
        """
        S = list(S)
        if K is None:
            K = len(S)

        # Check S before indexing so errors are clear
        self._check_open_set(S, K)

        # Deterministic tie-break: order S by candidate-column order, so the same
        # set of DCs always gives the same assignment regardless of input order.
        S_set = set(S)
        S_ord = [c for c in self.C.columns if c in S_set]

        sub_cost = self.C[S_ord]
        assignments = sub_cost.idxmin(axis=1)
        self._validate_solution(S, K, assignments)

        # ---- Delivery cost ----
        lane_costs = sub_cost.min(axis=1)
        wn = self.make_weights(w_obj)

        delivery_cost = float(lane_costs.sum())
        delivery_cost_weighted = float((lane_costs.values * wn).sum())
        delivery_obj = delivery_cost if w_obj is None else delivery_cost_weighted

        # ---- Storage + handling ----
        storage_cost = self.storage_cost(K)
        handling_cost = self.handling_cost
        total_modeled_cost = delivery_obj + storage_cost + handling_cost

        # ---- Assigned distances ----
        dc_idx = self._dist_cols.get_indexer(assignments.values)
        dist_assigned = pd.Series(
            self._dist_vals[np.arange(len(assignments)), dc_idx],
            index=assignments.index,
        )

        # Service metrics
        w = self.w_svc
        weighted_mean_km = float((w * dist_assigned).sum() / self.Q)

        order = dist_assigned.sort_values(kind="stable").index
        cum_ratio = (w.loc[order].cumsum() / self.Q).values
        idx_p95 = int(np.argmax(cum_ratio >= 0.95 - 1e-12))
        weighted_p95_km = float(dist_assigned.loc[order].iloc[idx_p95])

        cov_250 = float(w[dist_assigned <= 250].sum() / self.Q)
        cov_500 = float(w[dist_assigned <= 500].sum() / self.Q)
        max_dist = float(dist_assigned.max())

        metrics = {
            "k": K,
            "delivery_cost": delivery_cost,
            "delivery_cost_weighted": delivery_cost_weighted,
            "delivery_objective": delivery_obj,
            "storage_cost": storage_cost,
            "handling_cost": handling_cost,
            "total_modeled_cost": total_modeled_cost,
            "weighted_mean_km": weighted_mean_km,
            "weighted_p95_km": weighted_p95_km,
            "coverage_250": cov_250,
            "coverage_500": cov_500,
            "max_distance_km": max_dist,
        }

        # ---- Invariants ----
        assert np.isclose(total_modeled_cost, delivery_obj + storage_cost + handling_cost), \
            "Cost components don't reconcile"
        assert 0 <= cov_250 <= cov_500 <= 1 + 1e-12, \
            f"Coverage invalid: 250={cov_250}, 500={cov_500}"
        assert weighted_p95_km <= max_dist + 1e-9, \
            f"P95 ({weighted_p95_km}) > max dist ({max_dist})"

        if not quiet:
            print("\n[ COST BREAKDOWN (AUD) ]")
            print(f"  Delivery (unweighted) : {delivery_cost:,.3f}")
            print(f"  Delivery (weighted)   : {delivery_cost_weighted:,.3f}")
            print(f"  Storage (sqrt rule)   : {storage_cost:,.3f}")
            print(f"  Handling              : {handling_cost:,.3f}")
            print(f"  TOTAL MODELED COST    : {total_modeled_cost:,.3f}")
            print("\n[ SERVICE PERFORMANCE ]")
            print(f"  Mean Wt. Distance     : {weighted_mean_km:,.3f} km")
            print(f"  P95 Wt. Distance      : {weighted_p95_km:,.3f} km")
            print(f"  Max Assigned Distance : {max_dist:,.3f} km")
            print(f"  Coverage (<= 250km)   : {cov_250:.3%}")
            print(f"  Coverage (<= 500km)   : {cov_500:.3%}")

        return metrics, assignments


# ----------------------------------------------------------------------
# Initialization (cost_df, dist_df, branches, I_BAR, S_RATE, H_RATE come
# from earlier steps)
# ----------------------------------------------------------------------
evaluator = SharedEvaluator(
    cost_matrix=cost_df,
    dist_matrix=dist_df,
    demand_series=branches.set_index("DestinationRef")["total_cbm"],
    i_bar=I_BAR,
    s_rate=S_RATE,
    h_rate=H_RATE,
)

# ----------------------------------------------------------------------
# TEST 1: Smoke test
# ----------------------------------------------------------------------
print("\n>>> 1. SMOKE TEST (first 5 DCs) <<<")
top_5_dcs = cost_df.columns[:5].tolist()
res_5dc, _ = evaluator.evaluate(S=top_5_dcs, K=5, quiet=False)

# ----------------------------------------------------------------------
# TEST 2: Baseline unit test (K=1, NSW-DC01) - hard assert, tight tolerance
# ----------------------------------------------------------------------
print("\n>>> 2. BASELINE VALIDATION (K=1) <<<")
res_k1, _ = evaluator.evaluate(S=["NSW-DC01"], K=1)

targets = {
    "delivery_cost": 947_297.15,
    "storage_cost": 446_940.00,
    "handling_cost": 747_431.74,
    "total_modeled_cost": 2_141_668.89,
}

all_passed = True
for key, target_val in targets.items():
    actual_val = res_k1[key]
    ok = abs(actual_val - target_val) < 0.01
    all_passed &= ok
    print(f"  {'[PASS]' if ok else '[FAIL]'} {key:<20} Calc: {actual_val:>15,.2f} | Target: {target_val:>15,.2f}")
assert all_passed, "Baseline unit test FAILED - evaluator does not match EDA numbers"
print("  => Baseline verified.")

# ----------------------------------------------------------------------
# TEST 3: Storage cost vs FIXED expected values (not the same formula)
# ----------------------------------------------------------------------
print("\n>>> 3. INVENTORY POOLING RULE (sqrt K) CHECK <<<")
expected_storage = {
    1: 446_940.00,
    2: 632_068.61,
    3: 774_122.79,
    4: 893_880.00,
    5: 999_388.22,
    6: 1_094_774.95,
}
all_dcs = list(cost_df.columns)

print(f"  {'K':<4} | {'Computed':>15} | {'Expected':>15}")
print("  " + "-" * 42)
for k in range(1, 7):
    res_k, _ = evaluator.evaluate(S=all_dcs[:k], K=k)
    exp = expected_storage[k]
    print(f"  {k:<4} | {res_k['storage_cost']:>15,.2f} | {exp:>15,.2f}")
    assert abs(res_k["storage_cost"] - exp) < 1, f"Storage cost WRONG at K={k}"
print("  => Storage cost formula verified.")

# ----------------------------------------------------------------------
# TEST 4: weights never change assignment; total follows the scenario
# ----------------------------------------------------------------------
print("\n>>> 4. WEIGHT TEST (S3-style) <<<")
S3 = cost_df.columns[:3].tolist()
m_eq, a_eq = evaluator.evaluate(S3, K=3)
m_w, a_w = evaluator.evaluate(S3, K=3, w_obj=evaluator.q)

assert a_eq.equals(a_w), "Weights must not change assignment for a fixed S"
assert np.isclose(m_eq["total_modeled_cost"],
                  m_eq["delivery_cost"] + m_eq["storage_cost"] + m_eq["handling_cost"])
assert np.isclose(m_w["total_modeled_cost"],
                  m_w["delivery_cost_weighted"] + m_w["storage_cost"] + m_w["handling_cost"])
assert np.isclose(m_w["delivery_cost"], m_eq["delivery_cost"])  # unweighted part unchanged
print("  => Assignment identical; weighted total follows the scenario.")

# ----------------------------------------------------------------------
# TEST 5: small instance vs hand calculation (Section 15.3)
# ----------------------------------------------------------------------
print("\n>>> 5. SMALL-INSTANCE CHECK <<<")
sb = cost_df.index[:8]
sc = cost_df.columns[:4].tolist()
ev_small = SharedEvaluator(cost_df.loc[sb, sc], dist_df.loc[sb, sc],
                           evaluator.q.loc[sb], I_BAR, S_RATE, H_RATE)
m_s, _ = ev_small.evaluate(sc[:2], K=2)
manual = float(cost_df.loc[sb, sc[:2]].min(axis=1).sum())
assert np.isclose(m_s["delivery_cost"], manual), "Small instance mismatch"
print(f"  => Delivery = {manual:,.2f} matches hand calculation.")

# ----------------------------------------------------------------------
# TEST 6: fast_delivery agrees with evaluate; input order does not matter
# ----------------------------------------------------------------------
print("\n>>> 6. FAST PATH + ORDER-INVARIANCE CHECK <<<")
S4 = cost_df.columns[:4].tolist()
m_a, a_a = evaluator.evaluate(S4, K=4)
m_b, a_b = evaluator.evaluate(S4[::-1], K=4)
assert a_a.equals(a_b), "Assignment depends on the order of S"
assert np.isclose(evaluator.fast_delivery(S4), m_a["delivery_cost"])
m_c, _ = evaluator.evaluate(S4, K=4, w_obj=evaluator.q)
wn = evaluator.make_weights(evaluator.q)
assert np.isclose(evaluator.fast_delivery(S4, wn), m_c["delivery_objective"])
print("  => fast_delivery matches evaluate; assignment independent of S order.")
# ----------------------------------------------------------------------
# Save baseline + evaluator (V2 style)
# ----------------------------------------------------------------------
BASELINE_EVAL = res_k1          # lưu metrics của baseline K=1
EVALUATOR     = evaluator       # lưu evaluator để tái sử dụng ở BƯỚC 4+5

print("\n[ STEP 3 EVALUATOR SETUP COMPLETE ]\n")

STEP 3: SHARED EVALUATOR

>>> 1. SMOKE TEST (first 5 DCs) <<<

[ COST BREAKDOWN (AUD) ]
  Delivery (unweighted) : 749,842.587
  Delivery (weighted)   : 749,842.587
  Storage (sqrt rule)   : 999,388.222
  Handling              : 747,431.740
  TOTAL MODELED COST    : 2,496,662.549

[ SERVICE PERFORMANCE ]
  Mean Wt. Distance     : 738.475 km
  P95 Wt. Distance      : 3,096.955 km
  Max Assigned Distance : 3,545.064 km
  Coverage (<= 250km)   : 23.993%
  Coverage (<= 500km)   : 57.688%

>>> 2. BASELINE VALIDATION (K=1) <<<
  [PASS] delivery_cost        Calc:      947,297.15 | Target:      947,297.15
  [PASS] storage_cost         Calc:      446,940.00 | Target:      446,940.00
  [PASS] handling_cost        Calc:      747,431.74 | Target:      747,431.74
  [PASS] total_modeled_cost   Calc:    2,141,668.89 | Target:    2,141,668.89
  => Baseline verified.

>>> 3. INVENTORY POOLING RULE (sqrt K) CHECK <<<
  K    |        Computed |        Expected
  ------------------------------------------


In [13]:
import time
import itertools
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack
from scipy.optimize import milp, LinearConstraint, Bounds

# =============================================================================
# Helpers
# =============================================================================
def apply_weights(cost_matrix, weights=None):
    """
    Returns a numpy cost matrix (I x J), multiplied by branch weights.
    weights: pd.Series (index = branch) or None. Normalized to mean = 1.
    """
    C = cost_matrix.values.astype(float)
    if weights is None:
        return C

    w = weights.reindex(cost_matrix.index)
    if w.isna().any():
        raise ValueError("Weights are missing values for some branches.")

    w_mean = w.mean()
    if w_mean == 0:
        raise ValueError("Mean of weights cannot be zero.")

    w = (w / w_mean).values
    return C * w[:, None]

def check_K(K, J):
    if not (1 <= K <= J):
        raise ValueError(f"Invalid K={K}, must be between 1 and {J}.")

# =============================================================================
# 1. EXACT FIXED-P MILP SOLVER
# =============================================================================
class ExactMILPSolver:
    """
    min  sum_ij c_ij x_ij
    s.t. sum_j x_ij = 1     (each branch assigned to exactly 1 DC)
         x_ij <= y_j        (cannot assign to closed DC)
         sum_j y_j = K      (exactly K DCs are opened)
         x, y in {0,1}
    """
    def __init__(self, time_limit=900, mip_gap=0.0):
        self.time_limit = time_limit
        self.mip_gap = mip_gap
        self.solver_name = "SciPy HiGHS"

    def _build(self, I, J, K):
        nx = I * J
        n_var = nx + J

        # (1) sum_j x_ij = 1
        A_assign = csr_matrix(
            (np.ones(nx), (np.repeat(np.arange(I), J), np.arange(nx))),
            shape=(I, n_var))

        # (3) sum_j y_j = K
        A_card = csr_matrix(
            (np.ones(J), (np.zeros(J, dtype=int), nx + np.arange(J))),
            shape=(1, n_var))

        A_eq = vstack([A_assign, A_card]).tocsr()
        b_eq = np.append(np.ones(I), K)

        # (2) x_ij - y_j <= 0
        rows = np.repeat(np.arange(nx), 2)
        cols = np.empty(2 * nx, dtype=int)
        vals = np.empty(2 * nx)

        cols[0::2] = np.arange(nx)
        vals[0::2] = 1.0
        cols[1::2] = nx + np.tile(np.arange(J), I)
        vals[1::2] = -1.0

        A_link = csr_matrix((vals, (rows, cols)), shape=(nx, n_var))

        return [LinearConstraint(A_eq, b_eq, b_eq),
                LinearConstraint(A_link, -np.inf, 0)]

    def solve(self, cost_matrix, K, weights=None):
        C = apply_weights(cost_matrix, weights)
        I, J = C.shape
        check_K(K, J)
        ids = list(cost_matrix.columns)

        constraints = self._build(I, J, K)          # Build outside solver timer
        c_obj = np.concatenate([C.ravel(), np.zeros(J)])

        t0 = time.perf_counter()                    # Start solver timer
        res = milp(
            c=c_obj,
            constraints=constraints,
            integrality=np.ones(I * J + J),
            bounds=Bounds(0, 1),
            options={"time_limit": float(self.time_limit),
                     "mip_rel_gap": float(self.mip_gap),
                     "disp": False},
        )
        runtime = time.perf_counter() - t0

        if res.x is None:
            raise RuntimeError(f"MILP K={K}: No feasible solution found ({res.message}).")

        y = res.x[I * J:]
        selected = [ids[j] for j in np.where(y > 0.5)[0]]

        gap = getattr(res, "mip_gap", None)
        bound = getattr(res, "mip_dual_bound", None)
        gap = float(gap) if gap is not None and np.isfinite(gap) else np.nan
        bound = float(bound) if bound is not None and np.isfinite(bound) else np.nan

        if res.status == 0 and (np.isnan(gap) or gap <= 1e-9):
            status = "Optimal"
        elif res.status == 0:
            status = "Optimal within gap tolerance"
        else:
            status = "Time limit (feasible)"

        return {
            "method": "MILP", "K": K,
            "objective": float(res.fun),
            "selected_facilities": selected,
            "solver": self.solver_name,
            "status": status,
            "best_bound": bound,
            "optimality_gap": gap,
            "runtime_sec": runtime,
        }

# =============================================================================
# 2. HEURISTIC: GREEDY-ADD + ONE-SWAP (Vectorized)
# =============================================================================
class HeuristicSolver:
    def __init__(self, max_iter=10_000, tol=1e-9):
        self.max_iter = max_iter
        self.tol = tol

    def solve(self, cost_matrix, K, weights=None):
        C = apply_weights(cost_matrix, weights)
        I, J = C.shape
        check_K(K, J)
        ids = np.array(cost_matrix.columns)

        t0 = time.perf_counter()

        # Stage 1: Greedy-Add
        S = []
        cur_cost = np.full(I, np.inf)
        for _ in range(K):
            closed = np.setdiff1d(np.arange(J), S)
            Z = np.minimum(cur_cost[:, None], C[:, closed]).sum(axis=0)
            j = int(closed[np.argmin(Z)])
            S.append(j)
            cur_cost = np.minimum(cur_cost, C[:, j])

        greedy_obj = float(cur_cost.sum())
        obj = greedy_obj

        # Stage 2: One-Swap (Best-improvement)
        it = 0
        while it < self.max_iter:
            closed = np.setdiff1d(np.arange(J), S)
            if closed.size == 0:
                break

            threshold = obj - self.tol * max(1.0, abs(obj))   # Relative threshold
            best_move, best_z = None, threshold

            for pos, out_j in enumerate(S):
                rest = S[:pos] + S[pos + 1:]
                base = C[:, rest].min(axis=1) if rest else np.full(I, np.inf)
                Z = np.minimum(base[:, None], C[:, closed]).sum(axis=0)
                k = int(np.argmin(Z))

                if Z[k] < best_z:
                    best_z, best_move = float(Z[k]), (pos, int(closed[k]))

            if best_move is None:
                break

            S[best_move[0]] = best_move[1]
            obj = best_z
            it += 1

        return {
            "method": "Heuristic", "K": K,
            "objective": float(obj),
            "greedy_objective": greedy_obj,
            "selected_facilities": ids[S].tolist(),
            "swap_iterations": it,
            "runtime_sec": time.perf_counter() - t0,
        }

# =============================================================================
# 3. TESTS
# =============================================================================
def brute_force(matrix, K):
    best_obj, best_S = np.inf, None
    for subset in itertools.combinations(matrix.columns, K):
        z = matrix[list(subset)].min(axis=1).sum()
        if z < best_obj:
            best_obj, best_S = z, list(subset)
    return best_S, float(best_obj)

def run_tests(cost_df):
    milp_s = ExactMILPSolver(time_limit=60)
    heur_s = HeuristicSolver()

    # Test 1: MILP vs Brute Force (Small Instance)
    print("\n[Test 1] MILP vs Brute Force (12 x 7)")
    sub = cost_df.iloc[:12, :7]
    for K in (1, 2, 3):
        _, z_bf = brute_force(sub, K)
        r = milp_s.solve(sub, K)
        assert np.isclose(r["objective"], z_bf), f"MILP incorrect at K={K}"
        print(f"  K={K}: MILP={r['objective']:,.2f} brute={z_bf:,.2f} PASS")

    # Test 2: Heuristic vs Brute Force (Heuristic cannot be better than exact optimum)
    print("\n[Test 2] Heuristic >= optimum (small instance)")
    for K in (1, 2, 3):
        _, z_bf = brute_force(sub, K)
        h = heur_s.solve(sub, K)
        gap = 100 * (h["objective"] - z_bf) / z_bf
        assert h["objective"] >= z_bf - 1e-6, "Heuristic better than brute force => error"
        print(f"  K={K}: heur={h['objective']:,.2f} gap={gap:.4f}% PASS")

    # Test 3: Reported objective matching
    print("\n[Test 3] Reported objective matches recomputed from selected facilities")
    for K in (2, 3):
        h = heur_s.solve(sub, K)
        recomputed = sub[h["selected_facilities"]].min(axis=1).sum()
        assert np.isclose(h["objective"], recomputed)
    print("  PASS")

    # Test 4: Baseline test K=1
    print("\n[Test 4] K=1 full instance vs closed form")
    r1 = milp_s.solve(cost_df, 1)
    h1 = heur_s.solve(cost_df, 1)
    z1 = float(cost_df.sum().min())
    assert np.isclose(r1["objective"], z1) and r1["status"] == "Optimal"
    assert np.isclose(h1["objective"], z1)
    print(f"  closed={z1:,.2f} MILP={r1['objective']:,.2f} heur={h1['objective']:,.2f} PASS")

    # Test 5: Weighted objective validation
    print("\n[Test 5] Weighted objective")
    w = pd.Series(np.linspace(1, 3, len(sub)), index=sub.index)
    Cw = sub.mul(w / w.mean(), axis=0)
    _, z_bf = brute_force(Cw, 2)
    assert np.isclose(milp_s.solve(sub, 2, weights=w)["objective"], z_bf)
    print("  PASS")

    # Test 6: Full Instance comparison (MILP vs Heuristic)
    print(f"\n[Test 6] Full instance {cost_df.shape}")
    for K in (3, 6):
        h = heur_s.solve(cost_df, K)
        m = milp_s.solve(cost_df, K)
        assert h["objective"] >= m["objective"] - 1e-6
        gap = 100 * (h["objective"] - m["objective"]) / m["objective"]
        print(f"  K={K}: heur={h['objective']:,.2f} ({h['runtime_sec']:.3f}s) | "
              f"MILP={m['objective']:,.2f} ({m['runtime_sec']:.2f}s, {m['status']}) | gap={gap:.4f}%")

    print("\nALL TESTS PASSED SUCCESSFULLY!")

if __name__ == "__main__":
    if "cost_df" not in globals():
        raise RuntimeError("cost_df not found (DataFrame branch x candidate DC) — please run cost generation step first.")
    run_tests(cost_df)


[Test 1] MILP vs Brute Force (12 x 7)
  K=1: MILP=5,231.52 brute=5,231.52 PASS
  K=2: MILP=4,223.96 brute=4,223.96 PASS
  K=3: MILP=4,015.55 brute=4,015.55 PASS

[Test 2] Heuristic >= optimum (small instance)
  K=1: heur=5,231.52 gap=0.0000% PASS
  K=2: heur=4,223.96 gap=0.0000% PASS
  K=3: heur=4,015.55 gap=0.0000% PASS

[Test 3] Reported objective matches recomputed from selected facilities
  PASS

[Test 4] K=1 full instance vs closed form
  closed=310,176.30 MILP=310,176.30 heur=310,176.30 PASS

[Test 5] Weighted objective
  PASS

[Test 6] Full instance (635, 88)
  K=3: heur=218,897.24 (0.002s) | MILP=218,897.24 (3.51s, Optimal) | gap=0.0000%
  K=6: heur=178,393.49 (0.005s) | MILP=178,393.49 (3.70s, Optimal) | gap=-0.0000%

ALL TESTS PASSED SUCCESSFULLY!


In [14]:
# Cell kiểm tra session
names = ["cost_df", "dist_df", "branches", "dc_meta", "cost_curve",
         "I_BAR", "S_RATE", "H_RATE", "PROCESSED_DIR",
         "evaluator", "ExactMILPSolver", "HeuristicSolver"]
for n in names:
    print(f"  {n:20s}: {'✅' if n in globals() else '❌ THIẾU'}")

  cost_df             : ✅
  dist_df             : ✅
  branches            : ✅
  dc_meta             : ✅
  cost_curve          : ✅
  I_BAR               : ✅
  S_RATE              : ✅
  H_RATE              : ✅
  PROCESSED_DIR       : ✅
  evaluator           : ✅
  ExactMILPSolver     : ✅
  HeuristicSolver     : ✅


In [15]:
# ============================================================
# BUOC 6: RUN 54 EXPERIMENTS (S0-S7) - VERSION KHOP BUOC 1-5
# ============================================================
# S0: MILP + Heuristic, K=1..6  → 12 runs (RQ2 - method comparison)
# S1-S7: MILP only, K=1..6      → 42 runs (RQ3 - robustness)
# Tong: 54 runs
# ============================================================
import os
import time
import numpy as np
import pandas as pd

# ─── Check dependency thuc te tu BUOC 1-5 ───
required = ['cost_df', 'dist_df', 'branches', 'dc_meta', 'cost_curve',
            'I_BAR', 'S_RATE', 'H_RATE', 'PROCESSED_DIR',
            'evaluator', 'ExactMILPSolver', 'HeuristicSolver']
missing = [f for f in required if f not in globals()]
if missing:
    raise RuntimeError(f"Thieu: {missing}. Chay cac BUOC 1-5 truoc!")

print("=" * 70)
print("BUOC 6: RUN 54 EXPERIMENTS")
print("=" * 70)
print(f"  cost_df:   {cost_df.shape}")
print(f"  dist_df:   {dist_df.shape}")
print(f"  branches:  {len(branches)}")
print(f"  dc_meta:   {len(dc_meta)}")

# ─── Tao RESULTS_DIR (BUOC 1-2 chua tao) ───
RESULTS_DIR = os.path.join(PROCESSED_DIR, "..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"  RESULTS_DIR = {RESULTS_DIR}")

# ─── Constants ───
KS = range(1, 7)
CAND_VERSION = "SA4_maxCBM_v1"
CURRENT_DC_ID = "NSW-DC01"

# ─── State mapping (base + S6) ───
STATE_FALLBACK_BASE = {
    "Australian Capital Territory": "New South Wales",
    "South Australia": "Victoria",
    "Tasmania": "Victoria",
    "Northern Territory": "Queensland",
}
STATE_FALLBACK_S6 = {
    "Australian Capital Territory": "New South Wales",
    "South Australia": "Western Australia",
    "Tasmania": "Victoria",
    "Northern Territory": "Western Australia",
}

# ─── Extract cost_curve dict: state -> list[(mode, a, b, lo, hi)] ───
CURVE = {}
for r in cost_curve.itertuples(index=False):
    CURVE.setdefault(r.State, []).append(
        (r.Mode, float(r[2]), float(r[3]), float(r[4]), float(r[5])))
for s in CURVE:
    CURVE[s].sort(key=lambda x: x[3])

# ─── DC state lookup ───
DC_STATE = dc_meta["State"].to_dict()


# ═══════════════════════════════════════════════════════════
# build_cost: tao cost matrix voi fallback + scale + distance multiplier
# ═══════════════════════════════════════════════════════════
def build_cost(dist_df_, fallback, scale=1.0):
    """c_ij = scale * (alpha + beta * d_ij); curve theo DC state."""
    C = np.zeros(dist_df_.shape)
    for j, dc in enumerate(dist_df_.columns):
        st = fallback.get(DC_STATE[dc], DC_STATE[dc])
        bands = CURVE[st]
        d = dist_df_.values[:, j]
        last = bands[-1]
        ic = np.full_like(d, last[1])
        sl = np.full_like(d, last[2])
        for k, (mode, a, b, lo, hi) in enumerate(bands):
            is_last = (k == len(bands) - 1)
            m = (d >= lo) if is_last else ((d >= lo) & (d < hi))
            ic[m], sl[m] = a, b
        C[:, j] = scale * (ic + sl * d)
    return pd.DataFrame(C, index=dist_df_.index, columns=dist_df_.columns)


# ═══════════════════════════════════════════════════════════
# solve_milp / solve_heuristic: wrap Code 1 (class)
# ═══════════════════════════════════════════════════════════
def solve_milp(cost_df_, w_obj, K, time_limit=900, gap=0.0):
    solver = ExactMILPSolver(time_limit=time_limit, mip_gap=gap)
    r = solver.solve(cost_df_, K, weights=w_obj)
    return {
        "S": r["selected_facilities"],
        "objective": r["objective"],
        "runtime": r["runtime_sec"],
        "solver_status": r["status"],
        "best_bound": r["best_bound"],
        "solver_gap": r["optimality_gap"],
        "mip_gap_limit": gap,
        "solver": r["solver"],
    }


def solve_heuristic(cost_df_, w_obj, K, max_iter=10_000):
    solver = HeuristicSolver(max_iter=max_iter)
    r = solver.solve(cost_df_, K, weights=w_obj)
    return {
        "S": r["selected_facilities"],
        "objective": r["objective"],
        "runtime": r["runtime_sec"],
        "solver_status": "Heuristic",
        "best_bound": np.nan,
        "solver_gap": np.nan,
        "mip_gap_limit": np.nan,
        "solver": "Greedy-Add + One-Swap",
    }


# ═══════════════════════════════════════════════════════════
# evaluate: wrap SharedEvaluator de tra ve dict phang
# ═══════════════════════════════════════════════════════════
def evaluate(cost_df_, dist_df_, S, w_obj, w_svc, cbm_handling, K):
    """Goi SharedEvaluator voi cost/dist/w_svc phu hop scenario."""
    ev = evaluator  # dung evaluator goc
    # Neu cost/dist/w_svc khac base -> tao evaluator tam
    need_custom = (
        not cost_df_.equals(evaluator.C) or
        not dist_df_.equals(evaluator.D) or
        (w_svc is not None and not w_svc.equals(evaluator.w_svc))
    )
    if need_custom:
        ev = SharedEvaluator(
            cost_matrix=cost_df_,
            dist_matrix=dist_df_,
            demand_series=evaluator.q,
            i_bar=evaluator.I_bar,
            s_rate=evaluator.s,
            h_rate=evaluator.h,
            w_svc=w_svc if w_svc is not None else evaluator.w_svc,
            cbm_for_handling=cbm_handling,
        )
    metrics, assign = ev.evaluate(S=S, K=K, w_obj=w_obj)
    metrics["total_objective_cost"] = metrics["total_modeled_cost"]
    return metrics, assign


# ═══════════════════════════════════════════════════════════
# run_scenario
# ═══════════════════════════════════════════════════════════
def run_scenario(sid, cost_df_, dist_df_, w_svc, w_obj=None,
                 cbm_for_handling=None, methods=("milp",),
                 gap=0.005, time_limit=600):
    cbm_for_handling = (branches.set_index("DestinationRef")["total_cbm"]
                        if cbm_for_handling is None else cbm_for_handling)
    objective_type = ("normalized weighted proxy" if w_obj is not None
                      else "equal-frequency lane-cost proxy")

    print(f"\n{'=' * 70}")
    print(f"SCENARIO {sid}  (objective: {objective_type})")
    print(f"{'=' * 70}")

    rows, assigns = [], []

    for method in methods:
        for K in KS:
            if method == "milp":
                res = solve_milp(cost_df_, w_obj, K, time_limit, gap)
            else:
                res = solve_heuristic(cost_df_, w_obj, K)

            ev, assign = evaluate(cost_df_, dist_df_, res["S"], w_obj,
                                   w_svc, cbm_for_handling, K)

            rid = f"{sid}_{method}_K{K}"
            rows.append({
                "run_id": rid,
                "scenario_id": sid,
                "method": method,
                "k": K,
                "candidate_set_version": CAND_VERSION,
                "objective_type": objective_type,
                "delivery_cost": ev["delivery_cost"],
                "delivery_cost_weighted": ev["delivery_cost_weighted"],
                "storage_cost": ev["storage_cost"],
                "handling_cost": ev["handling_cost"],
                "total_modeled_cost": ev["total_modeled_cost"],
                "total_objective_cost": ev["total_objective_cost"],
                "weighted_mean_km": ev["weighted_mean_km"],
                "weighted_p95_km": ev["weighted_p95_km"],
                "coverage_250": ev["coverage_250"],
                "coverage_500": ev["coverage_500"],
                "max_distance_km": ev["max_distance_km"],
                "runtime_seconds": res["runtime"],
                "solver_status": res["solver_status"],
                "best_bound": res["best_bound"],
                "solver_gap": res["solver_gap"],
                "mip_gap_limit": res["mip_gap_limit"],
                "heuristic_gap_pct": np.nan,
                "facility_ids": ";".join(res["S"]),
                "solver": res["solver"],
            })

            assigns += [
                dict(run_id=rid, scenario_id=sid, method=method, k=K,
                     branch_id=i, dc_id=j)
                for i, j in assign.items()
            ]

            print(f"  {method:9s} K={K}: "
                  f"delivery={ev['delivery_cost']:>11,.0f} "
                  f"storage={ev['storage_cost']:>9,.0f} "
                  f"total={ev['total_modeled_cost']:>11,.0f} "
                  f"p95={ev['weighted_p95_km']:>6.0f}km "
                  f"({res['runtime']:.2f}s, {res['solver_status']})")

    df = pd.DataFrame(rows)

    # Heuristic gap (chi khi ca 2 methods)
    if set(methods) == {"milp", "heuristic"}:
        for K in KS:
            m = df[(df.method == "milp") & (df.k == K)].iloc[0]
            h = (df.method == "heuristic") & (df.k == K)
            if m["solver_status"] == "Optimal":
                zh = df.loc[h, "delivery_cost"].iloc[0]
                df.loc[h, "heuristic_gap_pct"] = (
                    100 * (zh - m["delivery_cost"]) / m["delivery_cost"]
                )

    df.to_csv(f"{RESULTS_DIR}/runs_{sid}.csv", index=False)
    pd.DataFrame(assigns).to_csv(f"{RESULTS_DIR}/assign_{sid}.csv", index=False)

    return df


# ═══════════════════════════════════════════════════════════
# run_all: 8 scenarios
# ═══════════════════════════════════════════════════════════
def run_all():
    out = []
    q_cbm = branches.set_index("DestinationRef")["total_cbm"]

    # S0: base
    print("\n" + "#" * 70)
    print("# S0: BASE SCENARIO (MILP + Heuristic)")
    print("#" * 70)
    out.append(run_scenario(
        "S0", cost_df, dist_df,
        w_svc=q_cbm, methods=("milp", "heuristic"),
        gap=0.0, time_limit=900
    ))

    # S1: cost +20%
    out.append(run_scenario(
        "S1", build_cost(dist_df, STATE_FALLBACK_BASE, 1.20), dist_df,
        w_svc=q_cbm, methods=("milp",), gap=0.005, time_limit=600
    ))

    # S2: cost -20%
    out.append(run_scenario(
        "S2", build_cost(dist_df, STATE_FALLBACK_BASE, 0.80), dist_df,
        w_svc=q_cbm, methods=("milp",), gap=0.005, time_limit=600
    ))

    # S3: normalized CBM-weighted
    out.append(run_scenario(
        "S3", cost_df, dist_df,
        w_svc=q_cbm, w_obj=q_cbm,
        methods=("milp",), gap=0.005, time_limit=600
    ))

    # S4: kg thay CBM -- nhung BUOC 1-2 chi co CBM -> tam dung CBM
    # Neu muon S4 dung kg, phai sua BUOC 1-2 aggregate total_kg.
    # O day check neu co total_kg thi dung, khong thi fallback CBM.
    if "total_kg" in branches.columns:
        q_kg = branches.set_index("DestinationRef")["total_kg"]
        out.append(run_scenario(
            "S4", cost_df, dist_df,
            w_svc=q_kg, w_obj=q_kg,
            cbm_for_handling=q_cbm,
            methods=("milp",), gap=0.005, time_limit=600
        ))
    else:
        print("\n[SKIP S4] branches chua co total_kg — chay S4 bang CBM (fallback)")
        out.append(run_scenario(
            "S4", cost_df, dist_df,
            w_svc=q_cbm, w_obj=q_cbm,
            cbm_for_handling=q_cbm,
            methods=("milp",), gap=0.005, time_limit=600
        ))

    # S5: demand QLD + WA +20%
    d5 = q_cbm.copy()
    bstate = branches.set_index("DestinationRef")["State"]
    mask = bstate.isin(["Queensland", "Western Australia"])
    d5.loc[mask[mask].index] *= 1.20
    out.append(run_scenario(
        "S5", cost_df, dist_df,
        w_svc=d5, w_obj=d5,
        cbm_for_handling=q_cbm,
        methods=("milp",), gap=0.005, time_limit=600
    ))

    # S6: doi mapping NT + SA
    out.append(run_scenario(
        "S6", build_cost(dist_df, STATE_FALLBACK_S6), dist_df,
        w_svc=q_cbm, methods=("milp",), gap=0.005, time_limit=600
    ))

    # S7: distance x1.20
    dist7 = dist_df * 1.20
    out.append(run_scenario(
        "S7", build_cost(dist7, STATE_FALLBACK_BASE), dist7,
        w_svc=q_cbm, methods=("milp",), gap=0.005, time_limit=600
    ))

    # Gom ket qua
    print("\n" + "=" * 70)
    print("GOM KET QUA")
    print("=" * 70)

    df_all = pd.concat(out, ignore_index=True)

    assert len(df_all) == 54, f"So runs = {len(df_all)}, ky vong 54"
    assert df_all.groupby("scenario_id").size().to_dict() == {
        "S0": 12, "S1": 6, "S2": 6, "S3": 6,
        "S4": 6, "S5": 6, "S6": 6, "S7": 6
    }, "So runs moi scenario sai!"

    df_all.to_csv(f"{RESULTS_DIR}/runs_all.csv", index=False)
    print(f"  Saved runs_all.csv ({len(df_all)} rows)")

    # Assignments
    assign_files = [f"{RESULTS_DIR}/assign_{sid}.csv"
                    for sid in ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7"]]
    pd.concat([pd.read_csv(f) for f in assign_files]) \
      .to_csv(f"{RESULTS_DIR}/assignments.csv", index=False)
    print(f"  Saved assignments.csv")

    # Facilities
    fac = []
    for _, r in df_all.iterrows():
        for dc in r["facility_ids"].split(";"):
            if dc in dc_meta.index:
                fac.append(dict(
                    run_id=r["run_id"], scenario_id=r["scenario_id"],
                    method=r["method"], k=r["k"], dc_id=dc,
                    state=dc_meta.loc[dc, "State"],
                    latitude=dc_meta.loc[dc, "Latitude"],
                    longitude=dc_meta.loc[dc, "Longitude"],
                ))
    pd.DataFrame(fac).to_csv(f"{RESULTS_DIR}/facilities.csv", index=False)
    print(f"  Saved facilities.csv")

    # K* summary
    milp_df = df_all[df_all.method == "milp"]
    kstar = milp_df.loc[
        milp_df.groupby("scenario_id")["total_modeled_cost"].idxmin(),
        ["scenario_id", "k", "total_modeled_cost",
         "weighted_p95_km", "facility_ids"]
    ].rename(columns={"k": "k_star"})
    kstar.to_csv(f"{RESULTS_DIR}/kstar_summary.csv", index=False)
    print(f"  Saved kstar_summary.csv")

    # In summary
    print(f"\n  Phan bo runs: {df_all.groupby('scenario_id').size().to_dict()}")

    print(f"\n{'=' * 70}")
    print("K* MOI SCENARIO (MILP)")
    print(f"{'=' * 70}")
    print(kstar.to_string(index=False))

    print(f"\n{'=' * 70}")
    print("S0: COST VS SERVICE THEO K")
    print(f"{'=' * 70}")
    s0 = df_all[(df_all.scenario_id == "S0") & (df_all.method == "milp")]
    print(s0[["k", "delivery_cost", "storage_cost", "handling_cost",
              "total_modeled_cost", "weighted_mean_km", "weighted_p95_km",
              "coverage_250", "coverage_500"]].round(2).to_string(index=False))

    print(f"\n{'=' * 70}")
    print("S0: HEURISTIC GAP")
    print(f"{'=' * 70}")
    h = df_all[(df_all.scenario_id == "S0") & (df_all.method == "heuristic")]
    print(h[["k", "heuristic_gap_pct", "runtime_seconds"]].round(4).to_string(index=False))

    return df_all, kstar


# ═══════════════════════════════════════════════════════════
# CHAY
# ═══════════════════════════════════════════════════════════
t_start = time.time()
df_all, kstar = run_all()
t_total = time.time() - t_start

print(f"\n{'=' * 70}")
print(f"BUOC 6 HOAN THANH")
print(f"{'=' * 70}")
print(f"  Tong runs:  {len(df_all)}")
print(f"  Runtime:    {t_total/60:.1f} phut")
print(f"  Output:     {RESULTS_DIR}/")

BUOC 6: RUN 54 EXPERIMENTS
  cost_df:   (635, 88)
  dist_df:   (635, 88)
  branches:  635
  dc_meta:   88
  RESULTS_DIR = ./processed/../results

######################################################################
# S0: BASE SCENARIO (MILP + Heuristic)
######################################################################

SCENARIO S0  (objective: equal-frequency lane-cost proxy)
  milp      K=1: delivery=    310,176 storage=  446,940 total=  1,504,548 p95=  2728km (28.45s, Optimal)
  milp      K=2: delivery=    256,687 storage=  632,069 total=  1,636,188 p95=  2728km (4.27s, Optimal)
  milp      K=3: delivery=    218,897 storage=  774,123 total=  1,740,452 p95=   777km (3.51s, Optimal)
  milp      K=4: delivery=    199,228 storage=  893,880 total=  1,840,540 p95=   679km (6.88s, Optimal)
  milp      K=5: delivery=    187,121 storage=  999,388 total=  1,933,941 p95=   679km (7.12s, Optimal)
  milp      K=6: delivery=    178,393 storage=1,094,775 total=  2,020,600 p95=   599km (4.11s